# --- TRAINING ---

In [ ]:
from google.colab import drive
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import torchvision.transforms as T
import torchvision.models as models
import torch.nn.functional as F
import tqdm


In [ ]:
# Montage du Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- CONFIGURATION DES CHEMINS ---
# Dossiers TRAIN :
IMG_TRAIN = "/content/drive/MyDrive/bdd10k/img/train"
MASK_TRAIN = "/content/drive/MyDrive/bdd10k/labels/train"

# Dossiers VAL :
IMG_VAL = "/content/drive/MyDrive/bdd10k/img/val"
MASK_VAL = "/content/drive/MyDrive/bdd10k/labels/val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Création de l'arborescence locale
!mkdir -p /content/local_bdd/img/train /content/local_bdd/img/val
!mkdir -p /content/local_bdd/labels/train /content/local_bdd/labels/val

#Transférer les données du drive en local sur colab
print("Copie des images (Train + Val)...")
!cp -r /content/drive/MyDrive/bdd10k/img/train/* /content/local_bdd/img/train/
!cp -r /content/drive/MyDrive/bdd10k/img/val/* /content/local_bdd/img/val/

print("Copie des masques (Train + Val)...")
!cp -r /content/drive/MyDrive/bdd10k/labels/train/* /content/local_bdd/labels/train/
!cp -r /content/drive/MyDrive/bdd10k/labels/val/* /content/local_bdd/labels/val/

print("Transfert terminé ! Tes données sont maintenant sur le SSD local.")

Copie des images (Train + Val)...
Copie des masques (Train + Val)...
Transfert terminé ! Tes données sont maintenant sur le SSD local.


In [ ]:
IMG_TRAIN = "/content/local_bdd/img/train"
MASK_TRAIN = "/content/local_bdd/labels/train"

IMG_VAL = "/content/local_bdd/img/val"
MASK_VAL = "/content/local_bdd/labels/val"

In [ ]:
LEARNING_RATE = 1e-4
BATCH_SIZE_NB = 16
EPOCHS = 20

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        # Upsample using ConvTranspose
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)

        # After concatenation, input channels = (upsampled_channels + skip_channels)
        combined_channels = (in_channels // 2) + skip_channels

        self.conv = nn.Sequential(
            nn.Conv2d(combined_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        # MobileNetV2 usually matches dimensions, but we cat anyway
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


In [ ]:
class MobileNetV2_UNet(nn.Module):
    def __init__(self, n_class):
        super().__init__()

        # 1. Load Pre-trained Backbone
        backbone = models.mobilenet_v2(weights='IMAGENET1K_V1').features

        # 2. Encoder: Extract specific layers for skip connections
        self.layer0 = backbone[0:2]   # Output: 16 channels, Size: 1/2 (112x112)
        self.layer1 = backbone[2:4]   # Output: 24 channels, Size: 1/4 (56x56)
        self.layer2 = backbone[4:7]   # Output: 32 channels, Size: 1/8 (28x28)
        self.layer3 = backbone[7:14]  # Output: 96 channels, Size: 1/16 (14x14)
        self.layer4 = backbone[14:18] # Output: 320 channels, Bottleneck, Size: 1/32 (7x7)

        # 3. Decoder: Defined to match the asymmetric channels of MobileNetV2
        # DecoderBlock(in_channels, skip_channels, out_channels)
        self.up1 = DecoderBlock(320, 96, 128) # Input 320, Skip 96 -> 128
        self.up2 = DecoderBlock(128, 32, 64)  # Input 128, Skip 32 -> 64
        self.up3 = DecoderBlock(64, 24, 32)   # Input 64, Skip 24 -> 32
        self.up4 = DecoderBlock(32, 16, 16)   # Input 32, Skip 16 -> 16

        # 4. Final Head: Map to number of classes
        # We need one last upsample to get from 112x112 back to original 224x224
        self.out_up = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(16, n_class, kernel_size=1)

    def forward(self, x):
        # Encoder path
        s0 = self.layer0(x)     # 16 channels
        s1 = self.layer1(s0)    # 24 channels
        s2 = self.layer2(s1)    # 32 channels
        s3 = self.layer3(s2)    # 96 channels
        x = self.layer4(s3)     # 320 channels (Bottleneck)

        # Decoder path with skip connections
        x = self.up1(x, s3)
        x = self.up2(x, s2)
        x = self.up3(x, s1)
        x = self.up4(x, s0)

        # Final upsampling to original input size
        x = self.out_up(x)
        return self.final_conv(x)


In [ ]:
# --- DATASET AVEC AUGMENTATIONS ---
class BDD10kDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=(384, 640), augment=False):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.size = size
        self.augment = augment

        raw_imgs = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        raw_masks = {f.replace('_train_id.png', ''): f for f in os.listdir(mask_dir) if f.endswith('.png')}

        self.valid_pairs = []
        for img_file in raw_imgs:
            img_id = img_file.split('.')[0]
            if img_id in raw_masks:
                self.valid_pairs.append((img_file, raw_masks[img_id]))

    def transform(self, image, mask):
        # 1. Resize obligatoire pour les deux
        image = TF.resize(image, self.size)
        mask = TF.resize(mask, self.size, interpolation=T.InterpolationMode.NEAREST)

        if self.augment:
            # Flip horizontal aléatoire (50% de chance)
            if random.random() > 0.5:
                image = TF.horizontal_flip(image)
                mask = TF.horizontal_flip(mask)

            # Color Jitter (Uniquement sur l'image, pas le masque !)
            color_jitter = T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2)
            image = color_jitter(image)

        # 2. Conversion en Tensor
        image = TF.to_tensor(image)
        mask = TF.to_tensor(mask)

        # 3. Normalisation ImageNet (Uniquement l'image)
        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        return image, mask

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.valid_pairs[idx]
        image = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, mask_name))

        mask_np = np.array(mask)
        road_mask = np.where(mask_np == 0, 1.0, 0.0).astype(np.float32)
        road_mask = Image.fromarray(road_mask)

        return self.transform(image, road_mask)

In [ ]:
# --- MÉTRIQUE IOU ---
def get_iou(preds, targets):
    preds = (torch.sigmoid(preds) > 0.5).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

In [ ]:
train_dataset = BDD10kDataset(IMG_TRAIN, MASK_TRAIN, augment=True) # On active l'augment ici
val_dataset = BDD10kDataset(IMG_VAL, MASK_VAL, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_NB, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_NB, shuffle=False)

In [ ]:
# 2. Instancier ton modèle (n_class=1 pour la route)
model = MobileNetV2_UNet(n_class=1).to(DEVICE)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 120MB/s]


In [ ]:

# 3. Fonction de perte et Optimiseur
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        intersection = (probs * targets).sum()
        dice = (2. * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1 - dice

def combined_loss(logits, targets):
    bce = nn.BCEWithLogitsLoss()(logits, targets)
    dice = DiceLoss()(logits, targets)
    return bce + dice

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3)

In [ ]:
def train_fn(model, loader, optimizer, criterion):
    model.train()
    loop = tqdm.tqdm(loader)
    total_loss = 0

    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)

def validate_fn(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
    return total_loss / len(loader)


In [ ]:
from torch.utils.tensorboard import SummaryWriter
import os

# Create directory for TensorBoard logs
TENSORBOARD_LOG_DIR = 'runs/segmentation_experiment'
os.makedirs(TENSORBOARD_LOG_DIR, exist_ok=True)
print(f"TensorBoard log directory created at: {TENSORBOARD_LOG_DIR}")

# Create directory for saving model checkpoints on Google Drive
MODEL_SAVE_DIR = '/content/drive/MyDrive/saved_models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
print(f"Model save directory created at: {MODEL_SAVE_DIR}")

# Initialize SummaryWriter
writer = SummaryWriter(TENSORBOARD_LOG_DIR)
print("SummaryWriter initialized.")

TensorBoard log directory created at: runs/segmentation_experiment
Model save directory created at: /content/drive/MyDrive/saved_models
SummaryWriter initialized.


In [ ]:
torch.cuda.empty_cache()
# --- LANCEMENT DE L'ENTRAÎNEMENT ---
global_step = 0
EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    loop = tqdm.tqdm(train_loader)

    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        outputs = model(images)
        loss = combined_loss(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # 🔹 Log batch loss
        writer.add_scalar("Loss/Train_batch", loss.item(), global_step)
        global_step += 1

        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    # 🔹 Log epoch train loss
    writer.add_scalar("Loss/Train_epoch", avg_train_loss, epoch)

    # Validation
    model.eval()
    val_loss = 0
    val_iou = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)

            val_loss += combined_loss(outputs, masks).item()
            val_iou += get_iou(outputs, masks).item()

    avg_val_loss = val_loss / len(val_loader)
    avg_iou = val_iou / len(val_loader)

    # 🔹 Log validation metrics
    writer.add_scalar("Loss/Validation", avg_val_loss, epoch)
    writer.add_scalar("Metrics/IoU", avg_iou, epoch)

    # 🔹 Log learning rate
    current_lr = optimizer.param_groups[0]["lr"]
    writer.add_scalar("Learning_Rate", current_lr, epoch)

    scheduler.step(avg_val_loss)

    print(
        f"📊 Epoch {epoch+1}: "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val IoU: {avg_iou:.4f}"
    )

    if (epoch + 1) % 5 == 0:
        torch.save(
            model.state_dict(),
            os.path.join(MODEL_SAVE_DIR, f"tesla_final_v1_epoch_{epoch+1}.pth")
        )

# Close TensorBoard writer
writer.close()

Epoch [1/20]: 100%|██████████| 438/438 [07:20<00:00,  1.01s/it, loss=0.565]


📊 Epoch 1: Train Loss: 0.9725 | Val Loss: 0.5898 | Val IoU: 0.8582


Epoch [2/20]: 100%|██████████| 438/438 [07:17<00:00,  1.00it/s, loss=0.224]


📊 Epoch 2: Train Loss: 0.4205 | Val Loss: 0.2500 | Val IoU: 0.8816


Epoch [3/20]: 100%|██████████| 438/438 [07:16<00:00,  1.00it/s, loss=0.797]


...

Epoch [10/20]: 100%|██████████| 438/438 [07:17<00:00,  1.00it/s, loss=0.153]


📊 Epoch 10: Train Loss: 0.1619 | Val Loss: 0.1490 | Val IoU: 0.8946


Epoch [11/20]: 100%|██████████| 438/438 [07:11<00:00,  1.01it/s, loss=0.109]


📊 Epoch 11: Train Loss: 0.1552 | Val Loss: 0.1536 | Val IoU: 0.8925


Epoch [12/20]:  40%|███▉      | 175/438 [02:51<04:00,  1.10it/s, loss=0.153]

# --- Inference ---

In [ ]:
# 1. Recréer l'architecture
model = MobileNetV2_UNet(n_class=1).to(DEVICE)

# 2. Charger les poids
model.load_state_dict(torch.load('model_epoch_5.pth', map_location=DEVICE))
model.eval() # Mode évaluation
print("Modèle chargé et prêt pour l'inférence.")

🚀 Modèle Tesla chargé et prêt pour l'inférence.


In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T

import time

def process_video(input_path, output_path, model):
    model.eval()
    cap = cv2.VideoCapture(input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_video, (width, height))

    transform = T.Compose([
        T.Resize((384, 640)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("🚗 Traitement en cours...")

    while cap.isOpened():
        start_time = time.time() # Début du chrono

        ret, frame = cap.read()
        if not ret: break

        # 1. Inférence
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        input_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            pred = torch.sigmoid(output).cpu().numpy()[0][0]
            mask = (pred > 0.5).astype(np.uint8)

        # 2. Post-processing & Overlay
        mask_hd = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
        blue_carpet = frame.copy()
        blue_carpet[mask_hd == 1] = [255, 100, 0]
        result_frame = cv2.addWeighted(blue_carpet, 0.4, frame, 0.6, 0)

        # 3. Calcul des FPS
        end_time = time.time()
        fps_current = 1 / (end_time - start_time)

        # 4. Affichage du texte FPS sur la frame
        cv2.putText(result_frame, f"FPS: {fps_current:.1f}", (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        out.write(result_frame)

    cap.release()
    out.release()
    print(f"✅ Terminé ! Vidéo sauvegardée sous : {output_path}")

process_video("video2.mp4", "resultat_tesla2.mp4", model)

🚗 Traitement en cours...
✅ Terminé ! Vidéo sauvegardée sous : resultat_tesla2.mp4


# --- YOLO ---

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.0 MB/s eta 0:00:00


In [ ]:
import cv2
import torch
import torch.nn as nn
import numpy as np
import time
from ultralytics import YOLO
from torchvision.models import mobilenet_v2

# --- 2. CONFIGURATION ET CHARGEMENT ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE = (640, 384) # (W, H) pour l'IA

# Chargement U-Net
unet_model = MobileNetV2_UNet(n_class=1).to(DEVICE)
unet_model.load_state_dict(torch.load('tesla_final_v1_epoch_10.pth', map_location=DEVICE))
unet_model.eval()

# Chargement YOLO
yolo_model = YOLO('yolov8n.pt')

# Pré-calcul des constantes de normalisation sur GPU
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

# --- 3. FONCTION DE TRAITEMENT ---
def process_tesla_vision(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) # On récupère le nombre total de frames

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_video, (width, height))

    print(f"🚀 Traitement lancé sur {DEVICE}...")

    # Initialisation de la barre de progression
    pbar = tqdm.tqdm(total=total_frames, desc="Analyse Tesla Vision", unit="frame")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        start_time = time.time()

        # --- ÉTAPE A : PRÉPARATION U-NET ---
        frame_resized = cv2.resize(frame, TARGET_SIZE)
        img_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)

        unet_input = torch.from_numpy(img_rgb).permute(2, 0, 1).float().to(DEVICE).unsqueeze(0)
        unet_input = unet_input.div(255.0).sub(MEAN).div(STD)

        # --- ÉTAPE B : INFÉRENCE DOUBLE ---
        with torch.no_grad():
            # 1. Segmentation
            unet_output = unet_model(unet_input)
            mask = (torch.sigmoid(unet_output) > 0.5).to(torch.uint8).cpu().numpy()[0][0]

            # 2. Détection YOLO (imgsz=320 pour la vitesse)
            # yolo_results = yolo_model(frame,persist=True, device=DEVICE, verbose=False,
            #                          classes=[0, 1, 2, 3, 5, 7], imgsz=320, conf=0.25)[0]
            yolo_results = yolo_model.track(frame, persist=True, device=DEVICE, verbose=False,
                               classes=[0, 1, 2, 3, 5, 7], imgsz=416, conf=0.2)[0]

        # --- ÉTAPE C : RENDU VISUEL ---
        mask_hd = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)

        # Coloration optimisée
        result_frame = frame.copy()
        result_frame[mask_hd == 1] = [255, 100, 0] # Tapis bleu
        result_frame = cv2.addWeighted(result_frame, 0.4, frame, 0.6, 0)

        # Dessin YOLO
        if yolo_results.boxes is not None:
            for box in yolo_results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls, conf = int(box.cls[0]), float(box.conf[0])

                cv2.rectangle(result_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(result_frame, f"{yolo_model.names[cls]} {conf:.2f}",
                            (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Calcul FPS moyen pour l'affichage
        fps_current = 1 / (time.time() - start_time)
        cv2.putText(result_frame, f"FPS: {fps_current:.1f}", (40, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        out.write(result_frame)

        # Mise à jour de la barre de progression
        pbar.update(1)
        pbar.set_postfix({"fps": f"{fps_current:.1f}"})

    pbar.close() # Fermeture propre
    cap.release()
    out.release()
    print(f"✨ Terminé ! Vidéo sauvegardée : {output_path}")

# --- 4. EXÉCUTION ---
process_tesla_vision("video2.mp4", "tesla_output_final.mp4")

🚀 Traitement lancé sur cuda...




Analyse Tesla Vision:   0%|          | 0/279 [01:29<?, ?frame/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...


Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 104ms
Prepared 1 package in 22ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect





Analyse Tesla Vision:   0%|          | 1/279 [00:01<05:04,  1.10s/frame]

Analyse Tesla Vision:   0%|          | 1/279 [00:01<05:04,  1.10s/frame, fps=1.0]

Analyse Tesla Vision:   1%|          | 2/279 [00:01<02:25,  1.90frame/s, fps=1.0]

Analyse Tesla Vision:   1%|          | 2/279 [00:01<02:25,  1.90frame/s, fps=10.4]

Analyse Tesla Vision:   1%|          | 3/279 [00:01<01:32,  3.00frame/s, fps=10.4]

Analyse Tesla Vision:   1%|          | 3/279 [00:01<01:32,  3.00frame/s, fps=13.1]

Analyse Tesla Vision:   1%|▏         | 4/279 [00:01<01:06,  4.11frame/s, fps=13.1]

Analyse Tesla Vision:   1%|▏         | 4/279 [00:01<01:06,  4.11frame/s, fps=13.1]

Analyse Tesla Vision:   2%|▏         | 5/279 [00:01<01:06,  4.11frame/s, fps=14.0]

Analyse Tesla Vision:   2%|▏         | 6/279 [00:01<00:46,  5.92frame/s, fps=14.0]

Analyse Tesla Vision:   2%|▏         | 6/279 [00:01<00:46,  5.92frame/s, fps=12.0]

Analyse Tesla Vision:   3%|▎         | 7/279 [00:01<00:41,  6.55frame/s, fps=12.0]

An

✨ Terminé ! Vidéo sauvegardée : tesla_output_final.mp4


### Tracking optimization


In [ ]:
import cv2
import torch
import torch.nn as nn
import numpy as np
import time
from ultralytics import YOLO

# --- 2. CONFIGURATION ET CHARGEMENT ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE = (640, 384) # (W, H) pour l'IA

# Chargement U-Net
unet_model = MobileNetV2_UNet(n_class=1).to(DEVICE)
unet_model.load_state_dict(torch.load('tesla_final_v1_epoch_10.pth', map_location=DEVICE))
unet_model.eval()

# Chargement YOLO
yolo_model = YOLO('yolov8n.pt')

# Pré-calcul des constantes de normalisation sur GPU
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

# --- 3. FONCTION DE TRAITEMENT ---
def process_tesla_vision(input_path, output_path):
    cap = cv2.VideoCapture(input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_video, (width, height))

    pbar = tqdm.tqdm(total=total_frames, desc="Analyse Tesla Vision", unit="frame")

    # Variables pour le saut de frame
    frame_count = 0
    yolo_results = None

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        start_time = time.time()

        # --- ÉTAPE A : SEGMENTATION (À chaque frame pour la fluidité) ---
        frame_resized = cv2.resize(frame, TARGET_SIZE)
        img_rgb = cv2.cvtColor(frame_resized, cv2.COLOR_BGR2RGB)
        unet_input = torch.from_numpy(img_rgb).permute(2, 0, 1).float().to(DEVICE).unsqueeze(0)
        unet_input = unet_input.div(255.0).sub(MEAN).div(STD)

        with torch.no_grad():
            unet_output = unet_model(unet_input)
            mask = (torch.sigmoid(unet_output) > 0.5).to(torch.uint8).cpu().numpy()[0][0]

        # --- ÉTAPE B : TRACKING OPTIMISÉ (1 frame sur 2) ---
        if frame_count % SKIP_FRAMES == 0 or yolo_results is None:
            # On utilise ByteTrack (plus rapide) et imgsz=320
            results = yolo_model.track(
                frame,
                persist=True,
                device=DEVICE,
                verbose=False,
                classes=[0, 1, 2, 3, 5, 7],
                imgsz=320,
                conf=0.2,
                tracker="bytetrack.yaml" # Tracker ultra-léger
            )
            yolo_results = results[0] if results else None

        # --- ÉTAPE C : RENDU VISUEL ---
        mask_hd = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)

        # Overlay du tapis bleu
        result_frame = frame.copy()
        result_frame[mask_hd == 1] = [255, 100, 0]
        result_frame = cv2.addWeighted(result_frame, 0.4, frame, 0.6, 0)

        # Dessin des boîtes (ID inclus si disponible)
        if yolo_results is not None and yolo_results.boxes is not None:
            for box in yolo_results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls = int(box.cls[0])
                conf = float(box.conf[0])

                # Récupération de l'ID de tracking
                id_txt = f"ID:{int(box.id[0])} " if box.id is not None else ""
                label = f"{id_txt}{yolo_model.names[cls]} {conf:.2f}"

                cv2.rectangle(result_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(result_frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # Affichage FPS
        fps_current = 1 / (time.time() - start_time)
        cv2.putText(result_frame, f"FPS: {fps_current:.1f}", (40, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        out.write(result_frame)
        pbar.update(1)
        pbar.set_postfix({"fps": f"{fps_current:.1f}"})

    pbar.close()
    cap.release()
    out.release()
    print(f"✨ Terminé ! Vidéo sauvegardée : {output_path}")

process_tesla_vision("video2.mp4", "tesla_output_final.mp4")



Analyse Tesla Vision:   0%|          | 0/279 [00:00<?, ?frame/s]

Analyse Tesla Vision:   0%|          | 1/279 [00:00<00:42,  6.57frame/s]

Analyse Tesla Vision:   0%|          | 1/279 [00:00<00:42,  6.57frame/s, fps=10.1]

Analyse Tesla Vision:   1%|          | 2/279 [00:00<00:42,  6.57frame/s, fps=29.9]

Analyse Tesla Vision:   1%|          | 3/279 [00:00<00:24, 11.35frame/s, fps=29.9]

Analyse Tesla Vision:   1%|          | 3/279 [00:00<00:24, 11.35frame/s, fps=23.3]

Analyse Tesla Vision:   1%|▏         | 4/279 [00:00<00:24, 11.35frame/s, fps=25.8]

Analyse Tesla Vision:   2%|▏         | 5/279 [00:00<00:21, 13.04frame/s, fps=25.8]

Analyse Tesla Vision:   2%|▏         | 5/279 [00:00<00:21, 13.04frame/s, fps=28.2]

Analyse Tesla Vision:   2%|▏         | 6/279 [00:00<00:20, 13.04frame/s, fps=24.1]

Analyse Tesla Vision:   3%|▎         | 7/279 [00:00<00:19, 14.22frame/s, fps=24.1]

Analyse Tesla Vision:   3%|▎         | 7/279 [00:00<00:19, 14.22frame/s, fps=33.8]

Analyse Tesla Visi

✨ Terminé ! Vidéo sauvegardée : tesla_output_final.mp4
